# R02 Probe Evaluation - retrieval-first graph shape

**Author**: Knowledge Graph Foundry autonomous build

Measures the pre-registered R02 hypotheses on the 28-probe CPAP set (`tests/probes/cpap-probe-set.yml`). Phase A ablates on the R01 graph: A0 baseline retrieval (propositions off), A1 propositions on (H11), A2 after similarity-edge densification (H13). Phase B (separate run) measures the H10/H12 rebuild graph. Scoring is deterministic: evidence recall by normalized substring match, answer-value match on numeric/value tokens, refusal regex for abstention.

In [ ]:
# imports
import json  # results persistence
import os  # env overrides
import re  # scoring
from pathlib import Path  # paths

import yaml  # probe set

from knowledge_graph_foundry import Foundry, load_settings  # engine under test

In [ ]:
# configuration
PROBES_PATH = Path('../tests/probes/cpap-probe-set.yml')  # gold probe set
REPORTS = Path('../reports')  # results output
PHASE = os.environ.get('KGF_EVAL_PHASE', 'A')  # A = R01 graph ablations, B = rebuild graph
NEO4J_URI = os.environ.get('KGF_EVAL_URI')  # optional graph override (phase B)
print(f'phase={PHASE} uri={NEO4J_URI or "(default env)"}')

In [ ]:
# probe set
probes = yaml.safe_load(PROBES_PATH.read_text())
print(len(probes), 'probes:', {c: sum(1 for p in probes if p['category'] == c)
                               for c in ('single_fact', 'comparison', 'multi_hop', 'unanswerable')})

In [ ]:
# scoring - deterministic, reproducible
REFUSAL = re.compile(
    r'no information|not (?:available|stated|specified|mentioned|provided)|'
    r'lacks|does not (?:contain|include|specify|state|provide|mention)|'
    r'cannot answer|unable to|unanswerable|no (?:data|details|answer)', re.I)

def _norm(s):
    return re.sub(r'\s+', ' ', s.casefold())

def evidence_recall(gold_evidence, context):
    if not gold_evidence:
        return None
    ctx = _norm(context)
    return sum(1 for g in gold_evidence if _norm(g) in ctx) / len(gold_evidence)

def value_tokens(gold_answer):
    return [t for t in re.findall(r'[\w.\-/]*\d[\w.\-/]*', gold_answer)]

def answer_correct(probe, answer):
    ans = _norm(answer)
    if probe['category'] == 'unanswerable':
        return bool(REFUSAL.search(answer))
    tokens = value_tokens(probe['gold_answer'])
    if tokens:
        hit = sum(1 for t in tokens if _norm(t) in ans)
        return hit >= max(1, len(tokens) // 2 + (len(tokens) % 2))
    return _norm(probe['gold_answer']) in ans

def false_refusal(probe, answer):
    return probe['category'] != 'unanswerable' and bool(REFUSAL.search(answer))

In [ ]:
# eval runner - one pass over all probes for a given foundry
def run_eval(foundry, label):
    rows = []
    for p in probes:
        context_lines, _ = foundry._retrieve_local(p['question'])
        context = '\n'.join(context_lines)
        result = foundry.query(p['question'])
        answer = result['answer']
        rows.append({
            'id': p['id'], 'category': p['category'],
            'evidence_recall': evidence_recall(p['gold_evidence'], context),
            'correct': answer_correct(p, answer),
            'false_refusal': false_refusal(p, answer),
            'context_chars': len(context),
            'answer': answer,
        })
        print(f"{p['id']} {p['category']:<13} recall="
              f"{rows[-1]['evidence_recall'] if rows[-1]['evidence_recall'] is not None else '-'} "
              f"correct={rows[-1]['correct']}")
    return {'label': label, 'rows': rows}

def summarize(result):
    rows = result['rows']
    answerable = [r for r in rows if r['category'] != 'unanswerable']
    unans = [r for r in rows if r['category'] == 'unanswerable']
    recalls = [r['evidence_recall'] for r in answerable if r['evidence_recall'] is not None]
    by_cat = {}
    for cat in ('single_fact', 'comparison', 'multi_hop'):
        cat_rows = [r for r in rows if r['category'] == cat]
        by_cat[cat] = sum(r['correct'] for r in cat_rows) / len(cat_rows) if cat_rows else None
    return {
        'label': result['label'],
        'evidence_recall': sum(recalls) / len(recalls) if recalls else 0,
        'answer_accuracy': sum(r['correct'] for r in answerable) / len(answerable),
        'accuracy_by_category': by_cat,
        'correct_refusal': sum(r['correct'] for r in unans) / len(unans) if unans else None,
        'false_refusal': sum(r['false_refusal'] for r in answerable) / len(answerable),
        'avg_context_chars': int(sum(r['context_chars'] for r in rows) / len(rows)),
    }

## Phase A - ablations on the R01 graph

A0: propositions disabled (R01 retrieval baseline). A1: propositions enabled (H11 lever). A2: after the similarity-edge pass (H13 lever; graph state change, run once).

In [ ]:
results = {}
settings = load_settings()
if NEO4J_URI:
    settings.neo4j.uri = NEO4J_URI

settings.graphrag.propositions_enabled = False
with Foundry(settings.model_copy(deep=True)) as f:
    results['A0_baseline'] = run_eval(f, 'A0_baseline')
print(summarize(results['A0_baseline']))

In [ ]:
settings.graphrag.propositions_enabled = True
with Foundry(settings.model_copy(deep=True)) as f:
    results['A1_propositions'] = run_eval(f, 'A1_propositions')
print(summarize(results['A1_propositions']))

In [ ]:
# H13: densify (idempotent), then re-run with propositions on
from knowledge_graph_foundry.graph.densify import add_similarity_edges
with Foundry(settings.model_copy(deep=True)) as f:
    created = add_similarity_edges(
        f.driver, f.settings.graphrag.vector_index_name,
        threshold=f.settings.graphrag.similarity_threshold,
        top_k=f.settings.graphrag.similarity_top_k)
    print('similarity edges created:', created)
    results['A2_densified'] = run_eval(f, 'A2_densified')
print(summarize(results['A2_densified']))

In [ ]:
# summary table + persist
import datetime
summaries = [summarize(r) for r in results.values()]
for s in summaries:
    print(f"{s['label']:<16} evrecall={s['evidence_recall']:.3f} "
          f"acc={s['answer_accuracy']:.3f} refusal_ok={s['correct_refusal']} "
          f"false_refusal={s['false_refusal']:.3f} ctx={s['avg_context_chars']}")
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S')
out = REPORTS / f'probe-eval-phase{PHASE}-{stamp}.json'
out.write_text(json.dumps({'summaries': summaries, 'results': results}, indent=2, default=str))
print('saved', out)